# Exercise Week 11 - Rumble



# 1. Install Rumble

## Setup the Spark cluster in Azure

### Create a cluster

1. Sign into the azure portal (portal.azure.com).
1. Search for "HDInsight clusters" using the search box at the top.
<img src="https://cloud.inf.ethz.ch/s/WxpMXB3Jz8SykMw/download" width="900">
1. Note that under the *Subscription* section, you might be prompted that the subscription is not registered:
<img src="https://cloud.inf.ethz.ch/s/gyTcQYKFCn3Yg6J/download" width="500">

  To fix this, follow the *Click here to register* link, and in the new page, search for *hdinsight*. Then select the *Microsoft.HDInsight* Provider and click the *Register* button.  
<img src="https://cloud.inf.ethz.ch/s/oHn9eyeZRP4LfZq/download" width="500">

1. Create a new resource group (for example: 'exercise08').
1. Give the cluster a unique name.
1. In the "Cluster Type" choose **Spark** and leave the default version as is. It is also indicated to use the **US West** region. 
1. Create a cluster login password (you can use https://www.random.org/strings/ for inspiration). Keep the password around as you will need it for later.
<img src="https://cloud.inf.ethz.ch/s/JY3DRLg8NLH559K/download" width="900">
1. Move to the *Storage* stage of the setup. Here, leave **Azure Storage** as the *Primary Storage Type*. For the *Primary Storage Account* you have the option to set up a new account. The *Container*'s name will be generated automatically, however make sure to remember it, or change it to something memorable, if you plan on finishing the exercises in more than one sitting.
<img src="https://cloud.inf.ethz.ch/s/NgtHE6iwSCZ8FQi/download" width="900">
1. Move to the *Configuration + Pricing* stage of the setup (skip *Security + networking*). Set up a Spark cluster which uses 2 **A5**  deployments as *Head* nodes and 2 **D12 v2** deployments for the *Worker* nodes. It should cost roughly 1.9 EUR/h. Note that if Azure allows you deploy more cores, then do so, by increasing the number of *Worker* nodes.
<img src="https://cloud.inf.ethz.ch/s/JpJEfjkZLPja5EK/download" width="900">
1. Move to the *Reivew + Create* stage of the setup, and click the **Create** button once validation succeeds.
1. Wait until your cluster is deployed (this can take up to 20 minutes).

<span style="color: red;">**Important:** Remember to **delete** the cluster once you are done. If you want to stop doing the exercises at any point, delete it and recreate it using the same container name as you used the first time, so that the resources are still there.</span>

<img src="https://cloud.inf.ethz.ch/s/2jLERoTD6q8nRMQ/download" width="900">

### Access your cluster

Make sure you can access your cluster (the NameNode) via SSH:

```
$ ssh <ssh_user_name>@<cluster_name>-ssh.azurehdinsight.net
```

If you are using Linux or MacOSX, you can use your standard terminal.
If you are using Windows you can use:
- Putty SSH Client and PSCP tool (get them at [here](http://www.chiark.greenend.org.uk/~sgtatham/putty/download.html)).
- This Notebook server terminal (Click on the Jupyter logo and the goto New -> Terminal).
- Azure Cloud Terminal (see the HBase exercise sheet for details)

You can access cluster's YARN in your browser
```
 https://<cluster_name>.azurehdinsight.net/yarnui/hn/cluster
```

**The cluster has its own Jupyter server.** We will use it. You can access it through the following link:
```
 https://<cluster_name>.azurehdinsight.net/jupyter
```

Access with username (by default ```admin```) and the password you chose.  



## Install Rumble

Then login to the shell and download the latest Rumble version:

```
wget https://github.com/RumbleDB/rumble/releases/download/v1.9.1/spark-rumble-1.9.1.jar
```

### HDInsight Shell

Unfortunately HDInsight will not provide us access to any other port than SSH.
Therefore the usual way to work with Rumble through HDInsight is through the shell.

### SSH Forwarding

Howeever for this sheet, we recommend to use SSH forwarding. For that, run:

```
spark-submit spark-rumble-1.9.1.jar --server yes --port 8002
```

and then on your local machine forward 8002 -> localhost:8002

```
ssh -N -L 8002:localhost:8002 sshuser@[servername]-ssh.azurehdinsight.net
```

For Windows users you would need to use a Unix terminal such as [Putty](https://www.putty.org/) to run these commands or use the Shell.

# 2. Setup Rumble in Jupyter Notebook

To get started, you first need to execute the cell below to activate the Rumble magic (you do not need to understand what it does, this is just initialization Python code).

In [ ]:
import requests
import json
import time
from IPython.core.magic import register_line_cell_magic

@register_line_cell_magic
def rumble(line, cell=None):
    if cell is None:
        data = line
    else:
        data = cell

    start = time.time()                                                         
    response = json.loads(requests.post(server, data=data).text)                   
    end = time.time()                                                              
    print("Took: %s ms" % (end - start))

    if 'warning' in response:
        print(json.dumps(response['warning']))
    if 'values' in response:
        for e in response['values']:
            print(json.dumps(e))
    elif 'error-message' in response:
        return response['error-message']
    else:
        return response

By default, this notebook uses a small public backend provided by us (very limited in CPU and memory, and with only the http scheme activated) that is sufficient to discover Rumble. This is new and experimental, so that it may occasionally break, especially if too many users use it at the same time, so please bear with us!

In order to use our backend, just execute the cell below.

In [ ]:
server='http://localhost:8002/jsoniq' # 'http://public.rumbledb.org:9090/jsoniq' public server in case you get stuck

It is straightforward to execute your own Rumble server on your own Spark cluster (and then you can make full use of all the input file systems and file formats).

## Install Juypter Notebook

Install Jupyter Notebook through this [link](https://jupyter.org/install.html) if you haven't yet. 

If you used **SSH Forwarding** you can now simply connect your colab to a local jupyter server by running 

```
jupyter notebook \
  --NotebookApp.allow_origin='https://colab.research.google.com' \
  --port=8888 \
  --NotebookApp.port_retries=0
```

on your local machine and connecting the kernel.

Now we are all set! You can now start reading and executing the JSONiq queries as you go, and you can even edit them!

# 3. Rumble Sandbox

## JSON

As explained on the [official JSON Web site](http://www.json.org/), JSON is a lightweight data-interchange format designed for humans as well as for computers. It supports as values:
- objects (string-to-value maps)
- arrays (ordered sequences of values)
- strings
- numbers
- booleans (true, false)
- null

JSONiq provides declarative querying and updating capabilities on JSON data.

## Elevator Pitch

JSONiq is based on XQuery, which is a W3C standard (like XML and HTML). XQuery is a very powerful declarative language that originally manipulates XML data, but it turns out that it is also a very good fit for manipulating JSON natively.
JSONiq, since it extends XQuery, is a very powerful general-purpose declarative programming language. Our experience is that, for the same task, you will probably write about 80% less code compared to imperative languages like JavaScript, Python or Ruby. Additionally, you get the benefits of strong type checking without actually having to write type declarations.
Here is an appetizer before we start the tutorial from scratch.


In [ ]:
%%rumble

let $stores :=
[
  { "store number" : 1, "state" : "MA" },
  { "store number" : 2, "state" : "MA" },
  { "store number" : 3, "state" : "CA" },
  { "store number" : 4, "state" : "CA" }
]
let $sales := [
   { "product" : "broiler", "store number" : 1, "quantity" : 20  },
   { "product" : "toaster", "store number" : 2, "quantity" : 100 },
   { "product" : "toaster", "store number" : 2, "quantity" : 50 },
   { "product" : "toaster", "store number" : 3, "quantity" : 50 },
   { "product" : "blender", "store number" : 3, "quantity" : 100 },
   { "product" : "blender", "store number" : 3, "quantity" : 150 },
   { "product" : "socks", "store number" : 1, "quantity" : 500 },
   { "product" : "socks", "store number" : 2, "quantity" : 10 },
   { "product" : "shirt", "store number" : 3, "quantity" : 10 }
]
let $join :=
  for $store in $stores[], $sale in $sales[]
  where $store."store number" = $sale."store number"
  return {
    "nb" : $store."store number",
    "state" : $store.state,
    "sold" : $sale.product
  }
return [$join]

## And here you go

### Actually, you already knew some JSONiq

The first thing you need to know is that a well-formed JSON document is a JSONiq expression as well.
This means that you can copy-and-paste any JSON document into a query. The following are JSONiq queries that are "idempotent" (they just output themselves):

In [ ]:
%%rumble
{ "pi" : 3.14, "sq2" : 1.4 }

In [ ]:
%%rumble
[ 2, 3, 5, 7, 11, 13 ]

In [ ]:
%%rumble
{
      "operations" : [
        { "binary" : [ "and", "or"] },
        { "unary" : ["not"] }
      ],
      "bits" : [
        0, 1
      ]
    }

In [ ]:
%%rumble
[ { "Question" : "Ultimate" }, ["Life", "the universe", "and everything"] ]


This works with objects, arrays (even nested), strings, numbers, booleans, null.

It also works the other way round: if your query outputs an object or an array, you can use it as a JSON document. JSONiq is a declarative language. This means that you only need to say what you want - the compiler will take care of the how.

In the above queries, you are basically saying: I want to output this JSON content, and here it is.

## JSONiq basics

### The real JSONiq Hello, World!

Wondering what a hello world program looks like in JSONiq? Here it is:

In [ ]:
%%rumble
"Hello, World!"

Not surprisingly, it outputs the string "Hello, World!".

### Numbers and arithmetic operations

Okay, so, now, you might be thinking: "What is the use of this language if it just outputs what I put in?" Of course, JSONiq can more than that. And still in a declarative way. Here is how it works with numbers:

In [ ]:
%%rumble
2 + 2

In [ ]:
%%rumble
 (38 + 2) div 2 + 11 * 2


(mind the division operator which is the "div" keyword. The slash operator has different semantics).

Like JSON, JSONiq works with decimals and doubles:

In [ ]:
%%rumble
 6.022e23 * 42

### Logical operations

JSONiq supports boolean operations.

In [ ]:
%%rumble
true and false

In [ ]:
%%rumble
(true or false) and (false or true)

The unary not is also available:

In [ ]:
%%rumble
not true

### Strings

JSONiq is capable of manipulating strings as well, using functions:


In [ ]:
%%rumble
concat("Hello ", "Captain ", "Kirk")

In [ ]:
%%rumble
substring("Mister Spock", 8, 5)

JSONiq comes up with a rich string function library out of the box, inherited from its base language. These functions are listed [here](https://www.w3.org/TR/xpath-functions-30/) (actually, you will find many more for numbers, dates, etc).



### Sequences

Until now, we have only been working with single values (an object, an array, a number, a string, a boolean). JSONiq supports sequences of values. You can build a sequence using commas:


In [ ]:
%%rumble
 (1, 2, 3, 4, 5, 6, 7, 8, 9, 10)

In [ ]:
%%rumble
1, true, 4.2e1, "Life"

The "to" operator is very convenient, too:

In [ ]:
%%rumble
 (1 to 100)

Some functions even work on sequences:

In [ ]:
%%rumble
sum(1 to 100)

In [ ]:
%%rumble
string-join(("These", "are", "some", "words"), "-")

In [ ]:
%%rumble
count(10 to 20)

In [ ]:
%%rumble
avg(1 to 100)

Unlike arrays, sequences are flat. The sequence (3) is identical to the integer 3, and (1, (2, 3)) is identical to (1, 2, 3).

## A bit more in depth

### Variables

You can bind a sequence of values to a (dollar-prefixed) variable, like so:

In [ ]:
%%rumble
let $x := "Bearing 3 1 4 Mark 5. "
return concat($x, "Engage!")

In [ ]:
%%rumble
let $x := ("Kirk", "Picard", "Sisko")
return string-join($x, " and ")

You can bind as many variables as you want:

In [ ]:
%%rumble
let $x := 1
let $y := $x * 2
let $z := $y + $x
return ($x, $y, $z)

and even reuse the same name to hide formerly declared variables:

In [ ]:
%%rumble
let $x := 1
let $x := $x + 2
let $x := $x + 3
return $x

### Iteration

In a way very similar to let, you can iterate over a sequence of values with the "for" keyword. Instead of binding the entire sequence of the variable, it will bind each value of the sequence in turn to this variable.

In [ ]:
%%rumble
for $i in 1 to 10
return $i * 2

More interestingly, you can combine fors and lets like so:

In [ ]:
%%rumble
let $sequence := 1 to 10
for $value in $sequence
let $square := $value * 2
return $square

and even filter out some values:

In [ ]:
%%rumble
let $sequence := 1 to 10
for $value in $sequence
let $square := $value * 2
where $square < 10
return $square

Note that you can only iterate over sequences, not arrays. To iterate over an array, you can obtain the sequence of its values with the [] operator, like so:


In [ ]:
%%rumble
[1, 2, 3][]

### Conditions

You can make the output depend on a condition with an if-then-else construct:

In [ ]:
%%rumble
for $x in 1 to 10
return if ($x < 5) then $x
                   else -$x

Note that the else clause is required - however, it can be the empty sequence () which is often when you need if only the then clause is relevant to you.

### Composability of Expressions

Now that you know of a couple of elementary JSONiq expressions, you can combine them in more elaborate expressions. For example, you can put any sequence of values in an array:

In [ ]:
%%rumble
[ 1 to 10 ]

Or you can dynamically compute the value of object pairs (or their key):

In [ ]:
%%rumble
{
      "Greeting" : (let $d := "Mister Spock"
                    return concat("Hello, ", $d)),
      "Farewell" : string-join(("Live", "long", "and", "prosper"),
                               " ")
}

You can dynamically generate object singletons (with a single pair):


In [ ]:
%%rumble
{ concat("Integer ", 2) : 2 * 2 }

and then merge lots of them into a new object with the {| |} notation:

In [ ]:
%%rumble
{|
    for $i in 1 to 10
    return { concat("Square of ", $i) : $i * $i }
|}

## JSON Navigation

Up to now, you have learnt how to compose expressions so as to do some computations and to build objects and arrays. It also works the other way round: if you have some JSON data, you can access it and navigate.
All you need to know is: JSONiq views
an array as an ordered list of values,
an object as a set of name/value pairs


### Objects

You can use the dot operator to retrieve the value associated with a key. Quotes are optional, except if the key has special characters such as spaces. It will return the value associated thereto:

In [ ]:
%%rumble
let $person := {
    "first name" : "Sarah",
    "age" : 13,
    "gender" : "female",
    "friends" : [ "Jim", "Mary", "Jennifer"]
}
return $person."first name"

You can also ask for all keys in an object:

In [ ]:
%%rumble
let $person := {
    "name" : "Sarah",
    "age" : 13,
    "gender" : "female",
    "friends" : [ "Jim", "Mary", "Jennifer"]
}
return { "keys" : [ keys($person)] }

### Arrays

The [[]] operator retrieves the entry at the given position:

In [ ]:
%%rumble
let $friends := [ "Jim", "Mary", "Jennifer"]
return $friends[[1+1]]

It is also possible to get the size of an array:

In [ ]:
%%rumble
let $person := {
    "name" : "Sarah",
    "age" : 13,
    "gender" : "female",
    "friends" : [ "Jim", "Mary", "Jennifer"]
}
return { "how many friends" : size($person.friends) }

Finally, the [] operator returns all elements in an array, as a sequence:

In [ ]:
%%rumble
let $person := {
    "name" : "Sarah",
    "age" : 13,
    "gender" : "female",
    "friends" : [ "Jim", "Mary", "Jennifer"]
}
return $person.friends[]

### Relational Algebra

Do you remember SQL's SELECT FROM WHERE statements? JSONiq inherits selection, projection and join capability from XQuery, too.

In [ ]:
%%rumble
let $stores :=
[
    { "store number" : 1, "state" : "MA" },
    { "store number" : 2, "state" : "MA" },
    { "store number" : 3, "state" : "CA" },
    { "store number" : 4, "state" : "CA" }
]
let $sales := [
    { "product" : "broiler", "store number" : 1, "quantity" : 20  },
    { "product" : "toaster", "store number" : 2, "quantity" : 100 },
    { "product" : "toaster", "store number" : 2, "quantity" : 50 },
    { "product" : "toaster", "store number" : 3, "quantity" : 50 },
    { "product" : "blender", "store number" : 3, "quantity" : 100 },
    { "product" : "blender", "store number" : 3, "quantity" : 150 },
    { "product" : "socks", "store number" : 1, "quantity" : 500 },
    { "product" : "socks", "store number" : 2, "quantity" : 10 },
    { "product" : "shirt", "store number" : 3, "quantity" : 10 }
]
let $join :=
    for $store in $stores[], $sale in $sales[]
    where $store."store number" = $sale."store number"
    return {
        "nb" : $store."store number",
        "state" : $store.state,
        "sold" : $sale.product
    }
return [$join]

### Access datasets

Rumble can read input from many file systems and many file formats. If you are using our backend, you can only use json-doc() with any URI pointing to a JSON file and navigate it as you see fit. 

In [ ]:
%%rumble
json-doc("Put any HTTP URL pointing to a JSON document here!").foo[[1]].bar.foobar[]

If you are using your own Rumble server on your cluster, you can also use any other function and scheme.

In [ ]:
%%rumble
json-file("put the path to a JSON lines file here. This will only work against your own Rumble backend and Spark cluster, though.")

# 4. The Great Language Game

This week you will be using again the [language confusion dataset](http://lars.yencken.org/datasets/languagegame/). You will write queries with Rumble. You will have to submit the results of this exercise to Moodle to obtain the weekly bonus. You will need four things:
- The query you wrote
- Something related to its output (which you will be graded on)
- The time it took you to write it
- The time it took you to run it

The execution time of the queries will be reported by Rumble.

Download and decompress the dataset in the same folder as `spark-rumble-1.9.1.jar` with the following:
```
wget http://data.greatlanguagegame.com.s3.amazonaws.com/confusion-2014-03-02.tbz2
tar -jxvf confusion-2014-03-02.tbz2
```



Afterwards upload the data into HDFS

```
hadoop dfs -copyFromLocal confusion-2014-03-02 /tmp/
```

## 4.1 Query the data

You can read data from a json file with `json-file`. For example, the following query will read and print the entries in the confusion dataset:




In [ ]:
%%rumble
for $i in json-file("/tmp/confusion-2014-03-02/confusion-2014-03-02.json")
return $i

Note that you have to press enter once at the end of each line and two more times to execute the query if you are using the **shell**. Your sparksoniq shell should look like this:
```
jiqs$ for $i in json-file("confusion-2014-03-02/confusion-2014-03-02.json")
>>> return $i
>>> 
>>> 
```



After the results of your query are printed, Rumble will report the execution runtime in milliseconds:
```
Took: 62.02618598937988 ms
```

In the `json-file` method you can optionally specify the number of partitions, which may allow your query to be parallelized and executed faster. For example:


In [ ]:
%%rumble
for $i in json-file("/tmp/confusion-2014-03-02/confusion-2014-03-02.json", 10)
return $i

## 4.2 SQL to Rumble

The following examples, show how SQL queries can be converted to Sparksoniq queries. Assume that the dataset is accessible with SQL through the table "entries".





### 4.2.1 Get all games played from Switzerland


```sql
SELECT *
FROM entries
WHERE country == "CH"
```


In [ ]:
%%rumble
...

### 4.2.2 Get all games played from Switzerland, where the correct answer (target) was "German"
```sql
SELECT *
FROM entries
WHERE country == "CH" AND target == "German"
```




In [ ]:
%%rumble
...

### 4.2.3 Get the top 5 games played from Switzerland, where the correct answer (target) was "German"
```sql
SELECT *
FROM entries
WHERE country == "CH" AND target == "German"
LIMIT 5
```




In [ ]:
%%rumble
...

### 4.2.4 Get all games played from Switzerland, where the correct answer (target) was "German", order them by date (ascending), and return the top 5 rows.
```sql
SELECT *
FROM entries
WHERE country == "CH" AND target == "German"
ORDER BY date ASC
LIMIT 5
```




In [ ]:
%%rumble
...

### 4.2.5 Get all games played from Switzerland, where the correct answer (target) was "German", group them by date, and return for each different date the number of games played.

```sql
SELECT date, COUNT(*) AS num_games
FROM entries
WHERE country == "CH" AND target == "German"
GROUP BY date
```


In [ ]:
%%rumble
...

### 4.2.6 Get all games played from Switzerland, group them by date and target, and return for each different date and target the number of games played.


NOTE: Rumble has some reserved keywords, for example `date`. If you try to create a variable `$date`, you may get an error, such as `no viable alternative at input 'date'`.

```sql
SELECT date, target, COUNT(*) AS num_games
FROM entries
WHERE country == "CH"
GROUP BY date, target
```




In [ ]:
%%rumble
...

### 4.2.7 For all games played from Switzerland, return the distinct targets of those games.

```sql
SELECT DISTINCT(target)
FROM entries
WHERE country == "CH"
```




In [ ]:
%%rumble
...

### 4.2.8 For all games played from Switzerland, get the distinct targets of those games, and return the index of "German" in the list of distinct targets.




In [ ]:
%%rumble
...


### 4.2.9 Count the number of games played from Switzerland (without any grouping).


NOTE: `distinct-values` and `index-of` work on "sequences". The method `json-file` returns a sequence. If you have an array on which you want to apply `distinct-values` and `index-of`, you must first convert it to a sequence. This can be done with `[]`. For example, if you have an array called `arr`, you can find its distinct values with `distinct-values(arr[])`

```sql
SELECT COUNT(*) AS count
FROM entries
WHERE country == "CH"
```


In [ ]:
%%rumble
...

## 4.3 More queries

Try writing a few more queries:
- List all chosen answers to games where the guessed language is correct (=target).

In [ ]:
%%rumble
...

- Count the games where the index of the correct answer in the choices array is 2 (as returned by the index-of method).

In [ ]:
%%rumble
...

- Return all games played on February 3rd 2014.

In [ ]:
%%rumble
...

# Moodle Graded Exercise

And now to the actual Moodle queries:


1\. Return the number of games where the guessed language is correct (=target), and that language is Russian.

In [ ]:
%%rumble

2\. Return the number of distinct values of languages (the target field).

In [ ]:
%%rumble

3\. Return the sample IDs (the sample field) of the top two games where the guessed language is correct (=target) ordered by language (ascending), then country (ascending), then date (ascending).

In [ ]:
%%rumble

4\. Aggregate all games by country and target language, counting the number of games for each pair (country, target). Return the highest three frequencies (counts).

In [ ]:
%%rumble

5\. Find the percentage of games where the guess was correct and it was the first choice amongst the array of possible answers.

In [ ]:
%%rumble

6\. Sort the languages by decreasing overall percentage of correct guesses and return the 3 first languages.

In [ ]:
%%rumble

7. Return the number games played on the latest day.

In [ ]:
%%rumble